In [ ]:
def iterate_x(Orientations_list, x_max):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (Orientations_list[i-1] == Orientations_list[i]
                and Orientations_list[i] == Orientations_list[i+1]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[i+1], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y(Orientations_list, x_max, y_max, pixel_misorientation_matrix):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[i + x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations(Orientations_list, x_max, y_max):
    pixel_misorientation_matrix = iterate_x(Orientations_list, x_max)
    pixel_misorientation_matrix = iterate_y(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix
    )
    return np.array(pixel_misorientation_matrix)

In [ ]:
def iterate_x_updated(Orientations_list, x_max, ix, iy):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (np.Orientations_list[i]  Orientations_list[ix + iy * x_max]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[ix + iy * x_max], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y_updated(Orientations_list, x_max, y_max, pixel_misorientation_matrix, ix, iy):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[ix + iy * x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations_updated(Orientations_list, x_max, y_max, ix, iy):
    pixel_misorientation_matrix = iterate_x_updated(Orientations_list, x_max, ix, iy)
    pixel_misorientation_matrix = iterate_y_updated(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix, ix, iy
    )
    return np.array(pixel_misorientation_matrix)

In [ ]:
test = grain_boundaries_misorientations(SPED8_oris, 200, 100)

plt.figure(figsize=(6,5))
img = plt.imshow(test)
plt.colorbar(img)
plt.title("Misorientations")
plt.show()

In [ ]:
test = grain_boundaries_misorientations_updated(SPED8_oris, 200, 100, 180, 5)

plt.figure(figsize=(6,5))
img = plt.imshow(test)
plt.colorbar(img)
plt.scatter(180, 5, s=10, c='black', marker="o")
plt.title("Misorientations")
plt.show()

In [ ]:
from skimage.draw import line

fig, ax = plt.subplots()
ax.imshow(test, cmap="viridis")

coords = []   # <-- this will store the pixel indices, not float coords
clicks = []   # temporary storage for float input

def onclick(event):
    if event.inaxes != ax:
        return
    
    # get float click coordinates
    x, y = event.xdata, event.ydata
    clicks.append((x, y))

    # plot click marker
    ax.plot(x, y, 'rx')
    fig.canvas.draw()

    # when two points clicked
    if len(clicks) == 2:
        (x0, y0), (x1, y1) = clicks
        
        # convert to integer pixel indices
        r0, c0 = int(y0), int(x0)
        r1, c1 = int(y1), int(x1)

        # compute all pixel indices on the line
        rr, cc = line(r0, c0, r1, c1)

        # save them to coords instead of printing
        coords.extend(list(zip(rr, cc)))

        print("Saved pixel indices in coords:")
        print(coords)

        # draw line on figure
        ax.plot([x0, x1], [y0, y1], 'r-')
        fig.canvas.draw()

        # reset clicks for next line
        clicks.clear()

fig.canvas.mpl_connect('button_press_event', onclick)
plt.show()

In [ ]:
value = []

for j in coords:
    value.append(test[j])

x_array = np.arange(0, len(value))

plt.figure()
plt.plot(x_array, value)
plt.show()